In [1]:
"""
Controle do início antecipado: compara o JRA do P02 / trial0122 rodado com a janela
original (pasta P02_trial0122_original) e com início 50 ms antes (P02_trial0122),
apenas nos quadros dentro da janela dos autores.
"""
import os
import numpy as np
import pandas as pd

PASTA = r'C:\JG\opensim\Resultados_FASE3'
ARQ = 'JRA_JointReaction_ReactionLoads.sto'


def ler(caminho):
    with open(caminho, errors='ignore') as f:
        for i, linha in enumerate(f):
            if 'endheader' in linha.lower():
                break
    df = pd.read_csv(caminho, sep=r'\s+', skiprows=i + 1, engine='python')
    df.columns = [c.strip() for c in df.columns]
    return df


def localizar_arquivo(tipo):
    esperado = os.path.join(PASTA, tipo, ARQ)
    if os.path.isfile(esperado):
        return esperado

    marcador_original = tipo.endswith('_original')
    for raiz, _, arquivos in os.walk(PASTA):
        caminho = os.path.join(raiz, ARQ)
        caminho_normalizado = raiz.lower().replace('-', '_')
        if (
            ARQ in arquivos
            and 'p02_trial0122' in caminho_normalizado
            and ('original' in caminho_normalizado) == marcador_original
        ):
            return caminho

    encontrados = [
        os.path.join(raiz, nome)
        for raiz, _, arquivos in os.walk(PASTA)
        for nome in arquivos
        if nome == ARQ
    ]
    raise FileNotFoundError(
        f'Não foi possível localizar o arquivo para {tipo!r} em {PASTA!r}. '
        f'Arquivos JRA encontrados: {encontrados}'
    )


orig = ler(localizar_arquivo('P02_trial0122_original'))
novo = ler(localizar_arquivo('P02_trial0122'))
novo = novo[novo['time'] >= orig['time'].min() - 1e-6]
comum = np.intersect1d(np.round(orig['time'], 5), np.round(novo['time'], 5))
o = orig[np.round(orig['time'], 5).isin(comum)].set_index(np.round(orig['time'], 5))
n = novo[np.round(novo['time'], 5).isin(comum)].set_index(np.round(novo['time'], 5))
base = 'knee_r_on_tibia_r_in_tibia_r_'
print(f'Quadros comparados: {len(comum)}')
for comp in ('mx', 'fx', 'fy', 'fz'):
    d = (n[base + comp] - o[base + comp]).abs()
    print(f'{comp}: diferença máxima {d.max():.4f} | pico original {o[base + comp].abs().max():.1f}')

FileNotFoundError: Não foi possível localizar o arquivo para 'P02_trial0122_original' em 'C:\\JG\\opensim\\Resultados_FASE3'. Arquivos JRA encontrados: ['C:\\JG\\opensim\\Resultados_FASE3\\P02_trial0122\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P02_trial0123\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P02_trial0130\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P03_trial0089\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P03_trial0095\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P03_trial0099\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P04_trial0079\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P04_trial0080\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P04_trial0081\\JRA_JointReaction_ReactionLoads.sto', 'C:\\JG\\opensim\\Resultados_FASE3\\P05_trial0